# 03 — Fundamental financial ratios

**Question:** Which companies show growth, profitability, cash generation and financial stability? Analysis uses information available by 31 December 2025, including provisional 2025 estimates. No investment recommendation is made.


## 1. Inputs and assumptions

Load approved cleaned data. Preserve missing sector-specific values; ratio functions never alter source amounts.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
project_root = Path.cwd().resolve()
if project_root.name == "notebooks": project_root = project_root.parent
if str(project_root) not in sys.path: sys.path.insert(0, str(project_root))
from src.data_loader import load_cleaned_data
from src.research.fundamentals import calculate_fundamentals, latest_fundamentals, compare_periods
annual, _ = load_cleaned_data(project_root / "data" / "Cleaned")
annual = annual.loc[annual.available_date.le(pd.Timestamp("2025-12-31"))].copy()
fundamentals = calculate_fundamentals(annual)
latest = latest_fundamentals(annual)
assert len(latest) == 20


## ⭐ 2. Ratio dictionary

Money amounts are GBP millions unless marked per share. Percentages below are displayed as decimal fractions in calculation tables; multiply by 100 for percentage points. Blank ratios mean undefined or inapplicable. A 0.01 materiality floor applies to positive monetary or EPS denominators.

| Measure | Formula | Unit | Interpretation |
|:--|:--|:--|:--|
| Revenue growth / CAGR | Current sales / prior sales - 1; CAGR uses an exact 3, 5 or 9-year root | fraction | Expansion from a positive base |
| Profit / EPS growth | Current / prior - 1, only with positive material values | fraction | Positive earnings growth |
| EPS | Profit / shares outstanding | GBP/share | Per-share earnings |
| Share-count growth | Current shares / prior shares - 1 | fraction | Positive means dilution |
| Profit / FCF margin | Profit or FCF / sales | fraction | Earnings or cash per sale |
| OCF / profit | Operating cash flow / positive profit | multiple | Earnings cash backing |
| Capex / revenue | Capital spending / sales | fraction | Reinvestment intensity |
| Net debt | Debt - cash | GBP m | Negative means net cash |
| Net debt / FCF or profit | Net debt / positive denominator | multiple | Repayment scale; excluded for CDB, BSL, CRM |
| Dividend payout | Dividend per share / positive EPS | fraction | Earnings distributed |
| Profit-margin stability | Five-year standard deviation | fraction | Historical margin variation |
| Loans overdue / capital cushion | Source fields for CDB and BSL | fraction | Credit stress and capital buffer |
| Claims ratio / capital coverage | Source fields for CRM | fraction / multiple | Underwriting pressure and capital cover |


## ⭐ 3. Growth and profitability

Profit and EPS growth are blank when either period is nonpositive or near zero. Read the underlying profit and EPS amounts alongside growth. Five-year mean and standard deviation describe margin level and stability.


In [ ]:
columns = ["ticker", "company", "sector", "sales_gbp_m", "revenue_growth", "revenue_cagr_3y", "revenue_cagr_5y", "revenue_cagr_9y", "profit_gbp_m", "profit_growth", "eps_gbp", "eps_growth", "profit_margin", "average_profit_margin_5y", "profit_margin_std_5y", "share_count_growth", "share_count_change_since_start"]
growth_profitability = latest[columns].copy()
display(growth_profitability.set_index("ticker").round(3))


### Sort the growth and profitability table

Choose any column shown in the table. Set `sort_header` to its exact column name and `high_to_low` to `True` for descending order or `False` for ascending order. Missing values stay at the bottom.


In [ ]:
# Edit these two settings to explore the comparison.
sort_header = "revenue_cagr_5y"
high_to_low = True

if sort_header not in growth_profitability.columns:
    raise ValueError(f"Choose a column from: {list(growth_profitability.columns)}")

sorted_growth_profitability = growth_profitability.sort_values(
    by=sort_header,
    ascending=not high_to_low,
    na_position="last",
)
display(sorted_growth_profitability.set_index("ticker").round(3))


## ⭐ 4. Cash generation and financial stability

Negative profit or FCF amounts remain visible; repayment multiples using them are blank. Lending and insurance capital require separate measures.


In [ ]:
columns = ["ticker", "sector", "operating_cash_flow_gbp_m", "capital_spending_gbp_m", "free_cash_flow_gbp_m", "fcf_margin", "ocf_to_profit", "capex_to_revenue", "debt_gbp_m", "cash_gbp_m", "net_debt_gbp_m", "net_debt_to_fcf", "net_debt_to_profit", "dividend_payout"]
display(latest[columns].set_index("ticker").round(3))


## 5. Sector-specific measures

CDB and BSL use loans overdue and capital cushion. CRM uses claims ratio and capital coverage. Changes in fraction measures are percentage points; capital coverage change is a multiple.


In [ ]:
fields = ["ticker", "year", "loans_overdue", "loans_overdue_change_pp", "capital_cushion", "capital_cushion_change_pp", "claims_ratio", "claims_ratio_change_pp", "capital_coverage"]
special = fundamentals.loc[fundamentals.ticker.isin(["CDB", "BSL", "CRM"]), fields].copy()
special["capital_coverage_change"] = special.groupby("ticker").capital_coverage.diff()
display(special.loc[special.year.ge(2021)].set_index(["ticker", "year"]).round(3))


## 6. Historical context

The sector panels show the FCF-margin history without placing 20 lines on one axis. Rising revenue with deteriorating FCF calls for further investigation.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharex=True)
for ax, (sector, subset) in zip(axes.flat, fundamentals.groupby("sector")):
    for ticker, series in subset.groupby("ticker"):
        ax.plot(series.year, series.fcf_margin * 100, label=ticker)
    ax.axhline(0, color="grey", linewidth=.7)
    ax.set(title=sector, ylabel="FCF margin (%)", xlabel="Fiscal year")
    ax.legend(fontsize=8, ncol=2)
axes.flat[-1].axis("off")
fig.suptitle("FCF margin by sector, 2016–2025")
fig.tight_layout()
plt.show()


In [ ]:
# A short evidence-led review queue; these labels are observations, not ratings.
prior = fundamentals.loc[fundamentals.year.eq(2024), ["ticker", "free_cash_flow_gbp_m"]].set_index("ticker")
review = latest[["ticker", "company", "revenue_growth", "profit_margin", "free_cash_flow_gbp_m", "net_debt_gbp_m"]].copy()
review["prior_fcf_gbp_m"] = review.ticker.map(prior.free_cash_flow_gbp_m)
review["revenue_up_fcf_down"] = review.revenue_growth.gt(0) & review.free_cash_flow_gbp_m.lt(review.prior_fcf_gbp_m)
print("Revenue rose but FCF fell in 2025 versus 2024 (investigate causes):")
display(review.loc[review.revenue_up_fcf_down].set_index("ticker").round(3))


## 7. Observations and limitations

**2025 examples from the review table:** FAI sales rose 34.5% year on year while FCF moved from −£141.24m to −£171.0m; HBL sales rose 3.1% while FCF fell from £66.81m to £50.4m. These are prompts to inspect investment spending and working capital, not evidence that the businesses will deteriorate.

The comparison tables show differences in growth and cash generation without combining them into a single score. The 2025 values are provisional; cash flow can move with working capital or investment timing. Review the factsheets to explain apparent divergence, especially where profit and FCF differ. Historical patterns are not forecasts.
